# GraphTrust large run: Regulated Finance

This CPU/high-RAM run generates and analyzes one 227,000-node, 2.5-million-edge SEIB-2026 enterprise graph. It writes an immutable run directory and a receipt that records whether execution occurred in Colab.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROFILE = "regulated_finance"
SEED = 2750159
REPO = "https://github.com/sronters/graph.git"
ROOT = Path("/content/graphtrust")
print(
    {
        "profile": PROFILE,
        "colab_release": os.getenv("COLAB_RELEASE_TAG"),
        "python": sys.version,
        "platform": platform.platform(),
        "cpus": os.cpu_count(),
    }
)

## 1. Locked environment and provenance

In [ ]:
if not (ROOT / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--all-extras"], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print({"git_commit": commit})

## 2. Capacity gate and large deterministic generation

In [ ]:
import psutil

available_gib = psutil.virtual_memory().available / 2**30
assert available_gib >= 10, f"Use a Colab high-RAM runtime; only {available_gib:.1f} GiB available"
dataset = ROOT / "data/generated" / PROFILE / "large" / str(SEED) / "injected_mixed"
if not (dataset / "checksums.sha256").exists():
    subprocess.run(
        [
            "uv",
            "run",
            "graphtrust",
            "generate",
            "--profile",
            PROFILE,
            "--scale",
            "large",
            "--seed",
            str(SEED),
            "--variants",
            "injected_mixed",
        ],
        check=True,
    )
print(dataset)

## 3. Immutable large GraphTrust run

In [ ]:
subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/run_large_profiles.py",
        "--profile",
        PROFILE,
        "--seed",
        str(SEED),
        "--config",
        "configs/large.yaml",
        "--output",
        "artifacts/large_runs",
    ],
    check=True,
)
receipt = Path("artifacts/large_runs/large_run_receipt.json")
print(receipt.read_text())

## 4. Verify checksums and export the complete evidence package

In [ ]:
import json

from google.colab import files

receipt_data = json.loads(Path("artifacts/large_runs/large_run_receipt.json").read_text())
assert receipt_data["is_google_colab"], "Receipt is not from a Google Colab runtime"
assert all(item["verified"] for item in receipt_data["profiles"])
archive = shutil.make_archive(
    f"/content/graphtrust-large-{PROFILE}-{SEED}",
    "zip",
    "artifacts/large_runs",
)
print(
    {
        "archive": archive,
        "verified_profiles": len(receipt_data["profiles"]),
    }
)
files.download(archive)